# Portfolio Optimization

**Prerequisites:** `05_portfolio/portfolio_construction_and_valuation.ipynb`.

This notebook demonstrates the configurable LP-based portfolio optimizer:
a flexible objective (maximize or minimize any metric expression),
attribute-based constraints, weight bounds, and turnover limits, solved for
optimal weights plus a generated trade list.

You author an optimization problem with the **typed builders** —
`PortfolioOptimizationSpec`, `Objective`, `MetricExpr`, `PerPositionMetric`,
`PositionFilter` and `Constraint` — and solve it with `optimize_portfolio_typed`.
The builders are the recommended authoring path: they are discoverable from an
IDE and they reject malformed specs at construction time rather than deep inside
the engine. Every spec still serializes to the same canonical JSON contract,
which the last section shows for callers that need the wire format directly.

In [1]:
import json

import pandas as pd

import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF, build_demo_market
from _shared.instrument_fixtures import fixed_bond
from finstack_quant.portfolio import (
    Constraint,
    Inequality,
    MetricExpr,
    MissingMetricPolicy,
    Objective,
    PerPositionMetric,
    PortfolioOptimizationSpec,
    PositionFilter,
    PortfolioValuation,
    WeightingScheme,
    optimize_portfolio,
    optimize_portfolio_typed,
    value_portfolio,
)
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import price_instrument_with_metrics

AS_OF = DEMO_AS_OF.isoformat()
print("Imports OK")


Imports OK


## Market context

`_shared.build_demo_market()` is the canonical example market used across this
curriculum. Only its `USD-OIS` discount curve matters here — every bond in the
portfolio discounts off it.

In [2]:
mc = build_demo_market()
market_json = mc.to_json()
print("Market context ready")

Market context ready


## Portfolio: five fixed-rate bonds

The bond payloads come from `_shared.instrument_fixtures.fixed_bond`, so this
notebook does not re-type a fixed-rate bond wire spec. Only the three fields
that carry the example's story — id, coupon and maturity — are overridden, so
coupons rise from 3% (AAA) to 8% (CCC) in line with typical credit spread tiers.

Each position carries `attributes` — a map of key/value pairs where values
can be strings (e.g. `"rating": "AAA"`) or numbers (e.g. `"credit_score": 800.0`).
The optimizer uses these attributes for indicator-based constraints and
filtered metric expressions.

In [3]:
def make_bond_position(idx, coupon, maturity, rating, credit_score):
    """Wrap the shared fixed-rate bond fixture as a portfolio position.

    The wire-level bond payload is `_shared.instrument_fixtures.fixed_bond`;
    only the id, coupon and maturity are overridden so that each bond lines up
    with its rating tier. `fixed_bond` returns a fresh dict per call, so
    mutating it here is safe.
    """
    bond_id = f"BOND-{rating}"
    _, instrument_spec = fixed_bond(idx)
    instrument_spec["spec"]["id"] = bond_id
    instrument_spec["spec"]["maturity"] = maturity
    instrument_spec["spec"]["cashflow_spec"]["Fixed"]["rate"] = str(coupon)
    return {
        "position_id": f"POS-{rating}",
        "entity_id": "FUND",
        "instrument_id": bond_id,
        "instrument_spec": instrument_spec,
        "quantity": 1.0,
        "unit": "units",
        "attributes": {
            "rating": rating,
            "sector": "corporate",
            "credit_score": credit_score,
        },
    }


positions = [
    make_bond_position(0, 0.03, "2029-01-15", "AAA", 800.0),
    make_bond_position(1, 0.04, "2027-01-15", "AA", 750.0),
    make_bond_position(2, 0.055, "2029-01-15", "BBB", 650.0),
    make_bond_position(3, 0.07, "2031-01-15", "BB", 550.0),
    make_bond_position(4, 0.08, "2029-01-15", "CCC", 400.0),
]

portfolio_spec = json.dumps({
    "id": "bond-fund",
    "as_of": AS_OF,
    "base_ccy": "USD",
    "entities": {"FUND": {"id": "FUND"}},
    "positions": positions,
})

print(f"Portfolio: {len(positions)} bonds")
for p in positions:
    rate = float(p["instrument_spec"]["spec"]["cashflow_spec"]["Fixed"]["rate"])
    print(f"  {p['position_id']:10s}  coupon={rate:.1%}  rating={p['attributes']['rating']}  score={p['attributes']['credit_score']}")


def verify_optimal_weights_and_ytm(optimal_weights: dict[str, float]) -> None:
    """Assert the budget constraint (weights sum to 1) and print engine YTM per position."""
    wsum = sum(optimal_weights.values())
    assert abs(wsum - 1.0) < 1e-5, f"optimal weights sum to {wsum}, expected 1.0"
    print(f"Constraint check: sum(optimal_weights) = {wsum:.12f}")
    by_pos = {p["position_id"]: p for p in positions}
    print("Per-position YTM (discounting, same market as optimization):")
    for pid in sorted(optimal_weights):
        inst_json = json.dumps(by_pos[pid]["instrument_spec"])
        out = price_instrument_with_metrics(
            inst_json, market_json, AS_OF, model="discounting", metrics=["ytm"]
        )
        ytm = ValuationResult.from_json(out).get_metric("ytm")
        if ytm is not None:
            print(f"  {pid}: YTM = {ytm:.4%}")
        else:
            print(f"  {pid}: YTM = (missing)")

Portfolio: 5 bonds
  POS-AAA     coupon=3.0%  rating=AAA  score=800.0
  POS-AA      coupon=4.0%  rating=AA  score=750.0
  POS-BBB     coupon=5.5%  rating=BBB  score=650.0
  POS-BB      coupon=7.0%  rating=BB  score=550.0
  POS-CCC     coupon=8.0%  rating=CCC  score=400.0


## Baseline valuation

Value the portfolio before optimization to see the starting state.

In [4]:
val_json = value_portfolio(portfolio_spec, market_json)
val = json.loads(val_json)

rows = [
    {
        "position": pv["position_id"],
        "pv": float(pv["value_base"]["amount"]),
        "currency": pv["value_base"]["currency"],
    }
    for pv in val.get("position_values", {}).values()
]

valuation = PortfolioValuation.from_json(val_json)
print(f"Total portfolio value: {valuation.total_value:,.2f} {valuation.base_ccy}")
pd.DataFrame(rows)

Total portfolio value: 5,213,035.26 USD


,position,pv,currency
0,POS-AAA,9.420210e+05,USD
1,POS-AA,9.855008e+05,USD
2,POS-BBB,1.032265e+06,USD
3,POS-BB,1.130740e+06,USD
4,POS-CCC,1.122509e+06,USD


## Example 1 — Basic yield maximization with rating cap

Maximize value-weighted YTM while constraining CCC exposure to at most 10%
of portfolio weight.

Read the builder chain from the inside out:

- `PerPositionMetric.metric("ytm")` — the per-position quantity the engine reads.
- `MetricExpr.value_weighted_average(...)` / `MetricExpr.weighted_sum(...)` — how
  those per-position values aggregate to one portfolio number.
- `Objective.maximize(...)` — the direction.
- `Constraint.metric_bound(expr, Inequality.le(), rhs)` — bound any metric
  expression. Here the expression is a weighted sum of
  `PerPositionMetric.attribute_indicator("rating", "eq", text="CCC")`, a 0/1
  indicator, so the bound reads directly as "CCC weight ≤ 10%".
- `Constraint.weight_bounds(PositionFilter.all(), 0.0, 0.40)` — per-position box
  constraints.

`PortfolioOptimizationSpec.new(...)` takes the portfolio and objective; the
`with_*` methods return a new spec each time, so specs compose without mutation.
The optimizer uses an LP formulation: weights are value-based shares that sum to
1.0 (fully invested budget constraint).

In [5]:
def rating_weight(rating: str) -> MetricExpr:
    """Portfolio weight carried by one rating bucket, as a metric expression."""
    return MetricExpr.weighted_sum(
        PerPositionMetric.attribute_indicator("rating", "eq", text=rating)
    )


# Objective: maximize the value-weighted average YTM across the book.
max_ytm = Objective.maximize(
    MetricExpr.value_weighted_average(PerPositionMetric.metric("ytm"))
)

spec1 = (
    PortfolioOptimizationSpec.new(portfolio_spec, max_ytm)
    .with_constraint(
        Constraint.metric_bound(rating_weight("CCC"), Inequality.le(), 0.10, label="ccc_cap")
    )
    .with_constraint(
        Constraint.weight_bounds(PositionFilter.all(), 0.0, 0.40, label="position_limits")
    )
    .with_weighting(WeightingScheme.value_weight())
    .with_missing_metric_policy(MissingMetricPolicy.zero())
    .with_label("basic_yield_max")
)

result1 = optimize_portfolio_typed(spec1, market_json)
print(f"Status:    {result1.status}")
print(f"Feasible:  {result1.is_feasible}")
print(f"Objective: {result1.objective_value:.4%}")
verify_optimal_weights_and_ytm(result1.optimal_weights)

Status:    OptimizationStatus.optimal(...)
Feasible:  True
Objective: 4.6720%
Constraint check: sum(optimal_weights) = 1.000000000000
Per-position YTM (discounting, same market as optimization):
  POS-AA: YTM = 4.7691%
  POS-AAA: YTM = 4.6035%
  POS-BB: YTM = 4.4898%
  POS-BBB: YTM = 4.6073%
  POS-CCC: YTM = 4.6108%


## Example 2 — Multi-constraint optimization

Because `with_constraint` returns a new spec, extending Example 1 is a matter of
chaining more constraints onto the same objective:

- CCC weight ≤ 10% (`Constraint.metric_bound` over an attribute indicator)
- BB weight ≤ 30% (same shape, different bucket)
- Max turnover ≤ 40% (`Constraint.max_turnover`)
- Each position weight in [0%, 40%] (`Constraint.weight_bounds`)

In [6]:
spec2 = (
    PortfolioOptimizationSpec.new(portfolio_spec, max_ytm)
    .with_constraint(
        Constraint.metric_bound(rating_weight("CCC"), Inequality.le(), 0.10, label="ccc_cap")
    )
    .with_constraint(
        Constraint.metric_bound(rating_weight("BB"), Inequality.le(), 0.30, label="bb_cap")
    )
    .with_constraint(Constraint.max_turnover(0.40, label="turnover_limit"))
    .with_constraint(
        Constraint.weight_bounds(PositionFilter.all(), 0.0, 0.40, label="position_limits")
    )
    .with_weighting(WeightingScheme.value_weight())
    .with_missing_metric_policy(MissingMetricPolicy.zero())
    .with_label("max_yield_constrained")
)

result = optimize_portfolio_typed(spec2, market_json)

print(f"Status:    {result.status}")
print(f"Feasible:  {result.is_feasible}")
print(f"Objective: {result.objective_value:.4%}")
print(f"Turnover:  {result.turnover:.2%}")
print(f"Binding:   {result.binding_constraints}")
verify_optimal_weights_and_ytm(result.optimal_weights)

Status:    OptimizationStatus.optimal(...)
Feasible:  True
Objective: 4.6544%
Turnover:  40.00%
Binding:   <built-in method binding_constraints of finstack_quant.portfolio.PortfolioOptimizationResult object at 0xa3c6c4000>
Constraint check: sum(optimal_weights) = 1.000000000000
Per-position YTM (discounting, same market as optimization):
  POS-AA: YTM = 4.7691%
  POS-AAA: YTM = 4.6035%
  POS-BB: YTM = 4.4898%
  POS-BBB: YTM = 4.6073%
  POS-CCC: YTM = 4.6108%


### Optimal weights and deltas

In [7]:
weights_df = pd.DataFrame([
    {
        "position": pos_id,
        "current_wt": result.current_weights.get(pos_id, 0.0),
        "optimal_wt": wt,
        "delta": result.weight_deltas.get(pos_id, 0.0),
    }
    for pos_id, wt in result.optimal_weights.items()
])
weights_df.style.format({"current_wt": "{:.2%}", "optimal_wt": "{:.2%}", "delta": "{:+.2%}"})

,position,current_wt,optimal_wt,delta
0,POS-AAA,18.07%,18.07%,-0.00%
1,POS-BB,21.69%,13.22%,-8.47%
2,POS-BBB,19.80%,19.80%,+0.00%
3,POS-CCC,21.53%,10.00%,-11.53%
4,POS-AA,18.90%,38.90%,+20.00%


### Trade list

`to_trade_list()` returns a sorted `list[TradeSpec]` with direction, quantities,
and weight changes.

In [8]:
trades = result.to_trade_list()
if trades:
    trades_df = pd.DataFrame([
        {
            "position": t.position_id,
            "instrument": t.instrument_id,
            "trade_type": str(t.trade_type),
            "direction": str(t.direction),
            "current_qty": t.current_quantity,
            "target_qty": t.target_quantity,
            "delta_qty": t.delta_quantity,
            "current_wt": t.current_weight,
            "target_wt": t.target_weight,
        }
        for t in trades
    ])
    display(trades_df)
else:
    print("No trades needed (portfolio already optimal).")

,position,instrument,trade_type,direction,current_qty,target_qty,delta_qty,current_wt,target_wt
0,POS-AA,BOND-AA,TradeType.existing(),TradeDirection.buy(),1.0,2.057946,1.057946,0.189045,0.389045
1,POS-CCC,BOND-CCC,TradeType.existing(),TradeDirection.sell(),1.0,0.464409,-0.535591,0.215327,0.100000
2,POS-BB,BOND-BB,TradeType.existing(),TradeDirection.sell(),1.0,0.609634,-0.390366,0.216906,0.132234


## Example 3 — Filtered metrics

`MetricExpr.weighted_sum` and `MetricExpr.value_weighted_average` both accept an
optional `filter`, which scopes the aggregation to a subset of positions. That
enables constraints like "the combined weight of BB and CCC positions must be at
least 20%" without affecting other positions.

Filters compose: `PositionFilter.and_`, `.or_`, `.not_`, `.by_attribute`,
`.by_entity_id` and `.by_position_ids`. Combined with
`PerPositionMetric.constant(1.0)`, a filtered weighted sum is exactly "the weight
of the selected bucket".

In [9]:
high_yield = PositionFilter.or_([
    PositionFilter.by_attribute("rating", "eq", text="BB"),
    PositionFilter.by_attribute("rating", "eq", text="CCC"),
])

hy_weight = MetricExpr.weighted_sum(PerPositionMetric.constant(1.0), filter=high_yield)

spec3 = (
    PortfolioOptimizationSpec.new(portfolio_spec, max_ytm)
    .with_constraint(
        Constraint.metric_bound(rating_weight("CCC"), Inequality.le(), 0.15, label="ccc_cap")
    )
    .with_constraint(
        Constraint.metric_bound(hy_weight, Inequality.ge(), 0.20, label="hy_weight_floor")
    )
    .with_constraint(
        Constraint.weight_bounds(PositionFilter.all(), 0.0, 0.50, label="position_limits")
    )
    .with_weighting(WeightingScheme.value_weight())
    .with_missing_metric_policy(MissingMetricPolicy.zero())
    .with_label("filtered_metrics_demo")
)

result3 = optimize_portfolio_typed(spec3, market_json)
print(f"Status:    {result3.status}")
print(f"Feasible:  {result3.is_feasible}")
print(f"Objective: {result3.objective_value:.4%}")
verify_optimal_weights_and_ytm(result3.optimal_weights)

Status:    OptimizationStatus.optimal(...)
Feasible:  True
Objective: 4.6829%
Constraint check: sum(optimal_weights) = 1.000000000000
Per-position YTM (discounting, same market as optimization):
  POS-AA: YTM = 4.7691%
  POS-AAA: YTM = 4.6035%
  POS-BB: YTM = 4.4898%
  POS-BBB: YTM = 4.6073%
  POS-CCC: YTM = 4.6108%


## The wire format

The typed builders are an *authoring* convenience: every spec serializes to one
canonical JSON contract, and `optimize_portfolio` solves that JSON directly.
Reach for the wire format when a spec is generated by another service, stored in
a config file, or authored from a language without the typed bindings.

The cell below prints the JSON that Example 1's `spec1` serializes to, then
hand-authors the same problem as raw JSON and checks that both routes produce the
same objective value. This is the one raw-JSON authoring example in the notebook
— prefer the builders everywhere else.

In [10]:
from finstack_quant.portfolio import (
    OptimizationStatus,
    PortfolioOptimizationResult,
    TradeSpec,
)

# 1. What the builders emit. `spec1.to_json()` is the canonical wire contract;
#    the embedded `portfolio` block is the same PortfolioSpec built earlier, so
#    it is dropped here to keep the printout readable.
spec1_wire = json.loads(spec1.to_json())
print("Spec keys:", list(spec1_wire.keys()))
print(json.dumps({k: v for k, v in spec1_wire.items() if k != "portfolio"}, indent=2))

# 2. The same problem hand-authored as raw JSON. Note the tagged-enum shapes:
#    objectives are {"Maximize": <MetricExpr>}, constraints are single-key maps
#    keyed by constraint kind, and enums are PascalCase strings.
raw_spec = {
    "portfolio": json.loads(portfolio_spec),
    "objective": {"Maximize": {"ValueWeightedAverage": {"metric": {"Metric": "ytm"}}}},
    "constraints": [
        {
            "MetricBound": {
                "label": "ccc_cap",
                "metric": {
                    "WeightedSum": {
                        "metric": {"AttributeIndicator": {"key": "rating", "op": "Eq", "value": "CCC"}}
                    }
                },
                "op": "Le",
                "rhs": 0.10,
            }
        },
        {"WeightBounds": {"label": "position_limits", "filter": "All", "min": 0.0, "max": 0.40}},
    ],
    "weighting": "ValueWeight",
    "missing_metric_policy": "Zero",
    "label": "basic_yield_max",
}

# 3. `optimize_portfolio` is the JSON entry point; `optimize_portfolio_typed` is
#    the builder entry point. Both drive the same Rust solver.
raw_result = json.loads(optimize_portfolio(json.dumps(raw_spec), market_json))
print(f"\nraw JSON objective  : {raw_result['objective_value']:.6%}")
print(f"typed spec objective: {result1.objective_value:.6%}")
assert abs(raw_result["objective_value"] - result1.objective_value) < 1e-12, (
    "raw JSON and typed builder must solve the same problem"
)
print("Both authoring routes agree.")

# 4. Round-trip: `from_json` rebuilds a typed spec from the wire payload, so a
#    spec authored here and one loaded from a config file are the same object.
typed_result = optimize_portfolio_typed(
    PortfolioOptimizationSpec.from_json(spec2.to_json()), market_json
)
assert isinstance(typed_result, PortfolioOptimizationResult)
assert isinstance(typed_result.status, OptimizationStatus)
assert all(isinstance(t, TradeSpec) for t in typed_result.to_trade_list())
print(f"Round-tripped spec2 objective: {typed_result.objective_value:.6%}"
      f" (direct: {result.objective_value:.6%})")

Spec keys: ['portfolio', 'objective', 'constraints', 'weighting', 'missing_metric_policy', 'label']
{
  "objective": {
    "Maximize": {
      "ValueWeightedAverage": {
        "metric": {
          "Metric": "ytm"
        }
      }
    }
  },
  "constraints": [
    {
      "MetricBound": {
        "label": "ccc_cap",
        "metric": {
          "WeightedSum": {
            "metric": {
              "AttributeIndicator": {
                "key": "rating",
                "op": "Eq",
                "value": "CCC"
              }
            }
          }
        },
        "op": "Le",
        "rhs": 0.1
      }
    },
    {
      "WeightBounds": {
        "label": "position_limits",
        "filter": "All",
        "min": 0.0,
        "max": 0.4
      }
    }
  ],
  "weighting": "ValueWeight",
  "missing_metric_policy": "Zero",
  "label": "basic_yield_max"
}

raw JSON objective  : 4.672019%
typed spec objective: 4.672019%
Both authoring routes agree.
Round-tripped spec2 objective: 4.

## Takeaways

Author with the typed builders; fall back to raw JSON only at a wire boundary.

| Builder | Role |
|---|---|
| `PortfolioOptimizationSpec.new(portfolio, objective)` | Root of the spec; `with_*` methods return a new spec, so specs compose without mutation |
| `Objective.maximize` / `.minimize` | Direction applied to a `MetricExpr` |
| `MetricExpr.value_weighted_average` / `.weighted_sum` | Aggregate per-position values into one portfolio number; both take an optional `filter` |
| `PerPositionMetric.metric` / `.attribute_indicator` / `.constant` / `.pv_base` | The per-position quantity the engine reads |
| `Constraint.metric_bound` / `.weight_bounds` / `.max_turnover` | Bound any metric expression, box per-position weights, or cap turnover |
| `PositionFilter.all` / `.by_attribute` / `.by_entity_id` / `.by_position_ids` / `.and_` / `.or_` / `.not_` | Scope metrics and constraints to position subsets |
| `WeightingScheme` / `MissingMetricPolicy` / `Inequality` | Enum constructors used across the spec |

| Entry point | Input |
|---|---|
| `optimize_portfolio_typed(spec, market)` | A `PortfolioOptimizationSpec` — the recommended path |
| `optimize_portfolio(spec_json, market)` | The same spec as canonical JSON, for specs generated elsewhere |

Both entry points drive the same Rust solver and return optimal weights, a trade
list, and binding-constraint diagnostics. Weights are value-based shares under a
fully-invested budget constraint (they sum to 1.0), and all constraint types are
composable.

## Appendix — the remaining `PositionFilter` constructors

Examples 1 and 3 used `PositionFilter.all()`, `.by_attribute` and `.or_`. The two
identity-based constructors are shown here against this notebook's own entity and
position ids, so every filter in the takeaways table has been constructed at least
once.

In [11]:
fund_only = PositionFilter.by_entity_id("FUND")
investment_grade = PositionFilter.by_position_ids(["POS-AAA", "POS-AA", "POS-BBB"])
not_ccc = PositionFilter.not_(PositionFilter.by_attribute("rating", "eq", text="CCC"))

print("by_entity_id('FUND')   :", fund_only)
print("by_position_ids(IG)    :", investment_grade)
print("not_(rating == 'CCC')  :", not_ccc)

# Any of these can scope a metric expression or a weight bound, e.g. cap the
# investment-grade sleeve at 80% of the book.
ig_cap = Constraint.metric_bound(
    MetricExpr.weighted_sum(PerPositionMetric.constant(1.0), filter=investment_grade),
    Inequality.le(),
    0.80,
    label="ig_cap",
)
result4 = optimize_portfolio_typed(
    PortfolioOptimizationSpec.new(portfolio_spec, max_ytm)
    .with_constraint(ig_cap)
    .with_constraint(
        Constraint.weight_bounds(PositionFilter.all(), 0.0, 0.40, label="position_limits")
    )
    .with_weighting(WeightingScheme.value_weight())
    .with_missing_metric_policy(MissingMetricPolicy.zero())
    .with_label("ig_cap_demo"),
    market_json,
)
print(f"\nWith an 80% IG cap -> status {result4.status}, objective {result4.objective_value:.4%}")

by_entity_id('FUND')   : PositionFilter.by_entity_id(...)
by_position_ids(IG)    : PositionFilter.by_position_ids(...)
not_(rating == 'CCC')  : PositionFilter.not(...)

With an 80% IG cap -> status OptimizationStatus.optimal(...), objective 4.6734%
